In [ ]:
import numpy as np
import gensim.downloader as api
from typing import List
from src.representations.tokenizer import Tokenizer   

class WordEmbedder:

    def __init__(self, model_name: str = "glove-wiki-gigaword-50"):
        print(f"Loading model: {model_name} ... This may take a while the first time.")
        self.model = api.load(model_name)
        self.vector_size = self.model.vector_size
        self.tokenizer = Tokenizer()

    def get_vector(self, word: str):
        if word in self.model:
            return self.model[word]
        return None

    def get_similarity(self, word1: str, word2: str):
        if word1 not in self.model or word2 not in self.model:
            return None
        return self.model.similarity(word1, word2)

    def get_most_similar(self, word: str, top_n: int = 10):
        if word not in self.model:
            return []
        return self.model.most_similar(word, topn=top_n)

    def embed_document(self, document: str):
        tokens = self.tokenizer.tokenize(document.lower())
        vectors = [self.model[t] for t in tokens if t in self.model]

        if not vectors:
            return np.zeros(self.vector_size)

        return np.mean(vectors, axis=0)



from src.representations.word_embedder import WordEmbedder

def main():
    we = WordEmbedder("glove-wiki-gigaword-50")

    print("=== Vector for 'king' ===")
    print(we.get_vector("king"))

    print("\n=== Similarity: king vs queen ===")
    print(we.get_similarity("king", "queen"))

    print("\n=== Similarity: king vs man ===")
    print(we.get_similarity("king", "man"))

    print("\n=== Top 10 similar to 'computer' ===")
    for w, score in we.get_most_similar("computer"):
        print(f"{w}: {score}")

    print("\n=== Document embedding ===")
    print(we.embed_document("The queen rules the country."))

if __name__ == "__main__":
    main()

import gensim
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

DATA_PATH = "data/UD_English-EWT/en_ewt-ud-train.txt"
SAVE_PATH = "results/word2vec_ewt.model"

def iter_sentences(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            tokens = simple_preprocess(line)
            if tokens:
                yield tokens

def main():
    print("Streaming dataset...")
    sentences = list(iter_sentences(DATA_PATH))

    print("Training Word2Vec model...")
    model = Word2Vec(
        sentences,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4,
        sg=1
    )

    print(f"Saving model to {SAVE_PATH}")
    model.save(SAVE_PATH)

    print("\nMost similar to 'computer':")
    if "computer" in model.wv:
        print(model.wv.most_similar("computer"))
    else:
        print("'computer' not in vocabulary")

    print("\nAnalogy test: king - man + woman = ?")
    try:
        print(model.wv.most_similar(positive=["king", "woman"], negative=["man"]))
    except:
        print("Some words missing.")

if __name__ == "__main__":
    main()


# ============================================
# test/lab4_spark_word2vec_demo.py
# ============================================
import re
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, split
from pyspark.ml.feature import Word2Vec

DATA_PATH = "data/c4-train.00000-of-01024-30K.json"

def main():
    spark = SparkSession.builder \
        .appName("Spark Word2Vec Demo") \
        .master("local[*]") \
        .getOrCreate()

    df = spark.read.json(DATA_PATH)

    df_clean = (
        df.select(lower(col("text")).alias("text"))
          .withColumn("text", regexp_replace("text", r"[^a-z\s]", " "))
          .withColumn("words", split(col("text"), r"\s+"))
    )

    word2vec = Word2Vec(
        vectorSize=100,
        minCount=5,
        inputCol="words",
        outputCol="embedding"
    )

    model = word2vec.fit(df_clean)

    print("\n=== Top 5 similar to 'computer' ===")
    model.findSynonyms("computer", 5).show()

    spark.stop()

if __name__ == "__main__":
    main()
